# CNN Classification of CIFAR-10

This notebook presents a simple CNN-based classification example for CIFAR-10.

The goal is educational practice only. It does not focus on architecture development or optimization. The code and workflow can be reused quickly across different scenarios or serve as an Agent tool.

This is public-interest educational code, assisted by DeepSeek and tested on SCNet.

Original reference: .................................................................................................

## 1. Download the dataset

If the network is restricted, downloading may take about one hour. You can also download the dataset offline from the official website.

Official CIFAR-10 archive: [cifar-10-python.tar.gz](https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import os

# ------------------------------
# 1. Download the dataset
# ------------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)

## 2. Inspect sample input data

Randomly select two examples from the training set and display them.

In [ ]:
# ------------------------------
# 2. Randomly select two examples and print them out
# ------------------------------
classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

def imshow(img, title=None):
    """Display a tensor image (C, H, W) as a normalised image."""
    img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    if title:
        plt.title(title)
    plt.show()

# Random indices
indices = np.random.choice(len(trainset), 2, replace=False)
for i, idx in enumerate(indices):
    image, label = trainset[idx]
    print(f"Sample {i+1}: Label = {classes[label]} (index {label})")
    imshow(image, title=classes[label])

## 3. Prepare the training data

Create PyTorch data loaders for training and testing.

In [ ]:
# ------------------------------
# 3. Prepare it for PyTorch (data loaders)
# ------------------------------
batch_size = 64
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

## 4. Define the CNN model

This example uses a classic 7-layer CNN automatically generated by DeepSeek. It does not involve architecture research or development.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    """
    A standard 7-layer CNN for CIFAR-10 (5 conv + 2 FC)
    Parameter count: ~1.2M
    Expected accuracy: ~85-90% with proper training (30-50 epochs)
    Architecture follows the classic VGG-style design pattern
    """
    def __init__(self):
        super(Net, self).__init__()

        # ========== Feature extraction: 5 convolutional layers ==========
        # Block 1: 3 -> 32 channels
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        # Block 2: 32 -> 64 channels
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        # Block 3: 64 -> 128 channels
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        # Block 4: 128 -> 256 channels (no pooling here)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)

        # Block 5: 256 -> 256 channels
        self.conv5 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(256)

        # Pooling layer (reused throughout)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # ========== Classification: 2 fully connected layers ==========
        # After 3 pooling operations: 32 -> 16 -> 8 -> 4
        # Flattened size = 256 * 4 * 4 = 4096
        self.fc1 = nn.Linear(256 * 4 * 4, 512)
        self.dropout1 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, 256)
        self.dropout2 = nn.Dropout(0.5)
        self.fc3 = nn.Linear(256, 10)  # Output: 10 classes

    def forward(self, x):
        # Block 1: 32x32 -> 16x16 (after pool)
        x = self.pool(F.relu(self.bn1(self.conv1(x))))

        # Block 2: 16x16 -> 8x8
        x = self.pool(F.relu(self.bn2(self.conv2(x))))

        # Block 3: 8x8 -> 4x4
        x = self.pool(F.relu(self.bn3(self.conv3(x))))

        # Block 4: 4x4 -> 4x4 (no pooling, keeps spatial size)
        x = F.relu(self.bn4(self.conv4(x)))

        # Block 5: 4x4 -> 4x4
        x = F.relu(self.bn5(self.conv5(x)))

        # Flatten: from (batch, 256, 4, 4) to (batch, 4096)
        x = x.view(x.size(0), -1)

        # Fully connected layers with dropout
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        x = self.fc3(x)

        return x



device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

net = Net().to(device)




# After defining net = Net().to(device)
total_params = sum(p.numel() for p in net.parameters())
trainable_params = sum(p.numel() for p in net.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 5. Train the model

In [ ]:
# ------------------------------
# 5. Train the model with CUDA
# ------------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

epochs = 10
best_acc = 0.0

for epoch in range(epochs):
    # ---------- Training phase ----------
    net.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        # Training accuracy on the training set (optional)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        if i % 100 == 99:
            print(f'[Epoch {epoch+1}, Batch {i+1}] loss: {running_loss / 100:.3f}')
            running_loss = 0.0

    train_acc = 100.0 * correct / total
    print(f'Epoch {epoch+1} finished. Training accuracy: {train_acc:.2f}%')

    # ---------- Validation phase ----------
    net.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in testloader:
            images, labels = data[0].to(device), data[1].to(device)
            outputs = net(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = 100.0 * correct / total
    print(f'Validation accuracy: {val_acc:.2f}%')

    # Save the best model
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(net.state_dict(), './best_cifar10_cnn.pth')
        print(f'Best model saved with accuracy {best_acc:.2f}%')

print('Finished Training')
print(f'Best validation accuracy: {best_acc:.2f}%')

## 6. Save the trained model parameters

In [ ]:
# ------------------------------
# 6. Save the model
# ------------------------------
PATH = './cifar10_cnn.pth'
torch.save(net.state_dict(), PATH)
print(f"Model saved to {PATH}")

## 7. Test the trained model on sample cases

Reload the saved model and evaluate it on five randomly selected test samples.

In [ ]:
# ------------------------------
# 7. Restart the kernel import and use it for 5 randomly selected samples
#    (Simulate a fresh start by reloading the model and evaluating on test samples)
# ------------------------------
# In a real notebook you would restart the kernel here.
# We simulate by reloading the model from disk.
net = Net().to(device)
net.load_state_dict(torch.load(PATH))
net.eval()   # set to evaluation mode

# Randomly select 5 test samples
test_indices = np.random.choice(len(testset), 5, replace=False)
print("\nTesting 5 randomly selected samples from test set:")

with torch.no_grad():
    for idx in test_indices:
        image, label = testset[idx]
        # Add batch dimension and move to device
        input_tensor = image.unsqueeze(0).to(device)
        output = net(input_tensor)
        _, predicted = torch.max(output, 1)
        predicted_class = classes[predicted.item()]

        print(f"True label: {classes[label]} (index {label}), "
              f"Predicted: {predicted_class} (index {predicted.item()})")

        # Show the image (optional)
        imshow(image, title=f"True: {classes[label]} | Pred: {predicted_class}")

## Conclusion

At this point, the CNN model for CIFAR-10 has been trained successfully.

For job opportunities, please contact: yucongcai_business@outlook.com  
For research-related collaboration, please contact: yucongcai_research@outlook.com